# House Prices Project — Full Outline

## Phase 1: EDA (complete)
- Load data, check shape/dtypes
- Target variable: SalePrice
  - .describe(), skew (1.88), log1p skew (0.12) — decided to train on log1p(SalePrice)
  - Histograms: raw vs log-transformed
- Missing values
  - Identified 19 columns with nulls, counted and sorted
  - Categorized against data_description.txt: documented "NA=no feature" vs genuinely missing
  - Verified assumptions with cross-checks (e.g. PoolQC vs PoolArea, MasVnrType vs MasVnrArea)
- Numeric correlation
  - Correlation of all numeric features vs SalePrice
  - Full correlation heatmap (features vs each other) — flagged multicollinearity candidates
- Categorical features vs SalePrice
  - Screened all 43 categorical columns (spread of avg price + category counts)
  - Verified with .value_counts() to catch false positives from tiny categories
  - Confirmed via boxplot: Neighborhood, ExterQual, KitchenQual, FireplaceQu
  - Rejected: RoofMatl, Condition2, Exterior1st, Exterior2nd (unreliable/noisy)
- Outlier check
  - GrLivArea vs SalePrice scatter plot
  - Identified 2 outlier rows (index 523, 1298) — large houses, low price, Partial sale condition

## Phase 2: Data Cleaning (next)
- Fill missing values, column by column:
  - Documented "NA=None": PoolQC, MiscFeature, Alley, Fence, FireplaceQu, 
    Garage(Type/Finish/Qual/Cond), Bsmt(Qual/Cond/Exposure/FinType1/FinType2)
  - Special case: MasVnrType (5 inconsistent rows filled with mode, rest "None"), 
    MasVnrArea (fillna 0)
  - Genuinely missing: LotFrontage (median by Neighborhood), GarageYrBlt (0 or YearBuilt),
    Electrical (mode)
- Drop the 2 outlier rows (523, 1298) identified in Phase 1
- Fix dtypes where needed (e.g. MSSubClass — numeric but actually categorical)
- Final check: df.isnull().sum() returns all zeros

## Phase 3: Feature Engineering
- Combine features: TotalSF, total bathrooms, etc.
- Age-based features: YrSold - YearBuilt, YrSold - YearRemodAdd
- Binary flags: HasPool, HasGarage, Has2ndFloor, etc.
- Ordinal-encode quality columns with natural order (Ex/Gd/TA/Fa/Po -> 4/3/2/1/0)
- Decide ordinal (encode as numbers) vs nominal (one-hot later) for remaining categoricals

## Phase 4: Preprocessing Pipeline
- Separate X (features) from y (log1p(SalePrice))
- Build a ColumnTransformer: scale numerics, one-hot remaining nominal categoricals
- Fit only on train, never on test (avoid leakage)
- Wrap in an sklearn Pipeline for reuse across CV and final predictions

## Phase 5: Modeling
- Baseline model (Ridge regression) + cross-validation for benchmark RMSE
- Try tree-based models (XGBoost, LightGBM)
- Compare CV scores, pick promising candidates
- Hyperparameter tuning on best candidate(s)

## Phase 6: Ensembling (optional)
- Average or stack predictions from multiple models

## Phase 7: Final Submission
- Refit best model/ensemble on full training data
- Predict on test set, expm1() to undo log transform
- Format and save as submission.csv matching sample_submission.csv structure
- Submit to Kaggle, check leaderboard score

## Understanding correlation — worked example

**Data:** 5 houses, Size (sq ft) vs Price ($)

| House | Size | Price |
|---|---|---|
| A | 800 | 100,000 |
| B | 1000 | 130,000 |
| C | 1200 | 160,000 |
| D | 1400 | 190,000 |
| E | 1600 | 220,000 |

**Step 1 — averages:** Size avg = 1200, Price avg = 160,000

**Step 2 — deviations from average (value - avg), for each column:**

| House | Size dev | Price dev |
|---|---|---|
| A | -400 | -60,000 |
| B | -200 | -30,000 |
| C | 0 | 0 |
| D | +200 | +30,000 |
| E | +400 | +60,000 |

**Part A — numerator: do the two columns move together?**
Multiply Size dev x Price dev, row by row, then sum:
(-400)(-60000) + (-200)(-30000) + 0 + (200)(30000) + (400)(60000) = 60,000,000
-> Every product is positive because Size and Price are always on the *same side*
of their own average together (both above, or both below).

**Part B — denominator: how spread out is each column on its own?**
Square each column's deviations *separately* (never mixing Size with Price), sum each:
- Size: 400² + 200² + 0² + 200² + 400² = 400,000
- Price: 60000² + 30000² + 0² + 30000² + 60000² = 9,000,000,000
Multiply the two totals, take the square root:
sqrt(400,000 x 9,000,000,000) = 60,000,000

**Final step — divide A by B:**
60,000,000 / 60,000,000 = 1.0 -> perfect positive correlation

**Key distinction:**
- Part A pairs the two columns together (X dev x Y dev, same row) -> captures the relationship
- Part B looks at each column in isolation (X squared alone, Y squared alone) -> 
  captures spread, used only to rescale Part A into the -1 to +1 range

This is exactly what `.corr()` does automatically for every column pair in a dataframe.


## Null Values

- PoolQC          1453 DONE
- MiscFeature     1406 DONE
- Alley           1369 DONE
- Fence           1179 DONE
- MasVnrType       872 DONE
- FireplaceQu      690 DONE
- LotFrontage      259 DONE
- GarageQual        81 DONE
- GarageFinish      81 DONE
- GarageType        81 DONE
- GarageYrBlt       81 DONE
- GarageCond        81 DONE
- BsmtFinType2      38 DONE
- BsmtExposure      38 DONE
- BsmtCond          37 DONE
- BsmtQual          37 DONE
- BsmtFinType1      37 DONE
- MasVnrArea         8 DONE
- Electrical         1 DONE